# 02 · Add a Business Semantic Layer and Map It to the Physical Metadata

Run this **after** `01_load_metadata_into_graph.ipynb` succeeded.

Notebook 01 gave the graph the *physical* layer: tables, columns and sampled values exactly as Snowflake stores them. That tells an agent *where* the data is, not *what it means*. This notebook adds the *semantic* layer on top:

1. Checks that the physical layer from notebook 01 is in the graph
2. Looks at the business semantic model for the TPC-H data (`datasets/tpch_semantic_model.yaml`, in the Open Semantic Interchange (OSI) format)
3. Previews and ingests it with `neocarta osi ingest`
4. Proves it attached to the **existing** `Table` and `Column` nodes instead of creating a parallel copy
5. Explores what it added: join paths, business-term synonyms and metric definitions
6. Shows how the same retrieval tool an agent uses now returns richer context

> ⚠️ **Section 4 writes to Neo4j** (database `NEO4J_DATABASE`). Nothing is read from or written to Snowflake. Re-running is safe: ingestion is idempotent.

## 1 · Configuration

In [ ]:
import json, os, re, subprocess, sys
from pathlib import Path

import yaml
from dotenv import load_dotenv

env_file = next((p for p in (Path(".env"),) if p.exists()), None)
assert env_file, "No .env found: copy .env.example to .env and fill it in (see docs/attendee_guide.md)"
load_dotenv(env_file, override=True)
print("Config file:", env_file)

PLACEHOLDERS = ("xxxxxxxx", "your-", "xy12345", "workshop_user")
required = ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "SNOWFLAKE_DATABASE", "SNOWFLAKE_SCHEMA"]
bad = [k for k in required if not os.getenv(k) or any(p in os.getenv(k) for p in PLACEHOLDERS)]
assert not bad, f"Missing or placeholder values in {env_file}: {bad}. Run notebook 0a first."

NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
SF_DATABASE, SF_SCHEMA = os.environ["SNOWFLAKE_DATABASE"], os.environ["SNOWFLAKE_SCHEMA"]
SPEC = Path("datasets/tpch_semantic_model.yaml")
assert SPEC.exists(), f"{SPEC} not found: run this notebook from the repository root"
print(f"Physical layer : {SF_DATABASE}.{SF_SCHEMA} (loaded by notebook 01)")
print(f"Semantic layer : {SPEC}")
print(f"Target         : Neo4j {os.environ['NEO4J_URI']} (database '{NEO4J_DATABASE}')")

### Options you can change

| Setting | Meaning |
|---|---|
| `INGEST_EMBEDDINGS` | `True` also embeds the new semantic nodes (metrics, business terms...) so vector search covers them. Needs a working `EMBEDDING_MODEL`; use the same one as notebook 01. |

> The semantic model is written for the TPC-H sample data. If you changed `SNOWFLAKE_SCHEMA` in notebook 01, this notebook does not apply to your schema (the YAML would have to describe *your* tables).

In [ ]:
INGEST_EMBEDDINGS = True

import litellm

embedding_model = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
embedding_ok = False
if INGEST_EMBEDDINGS:
    try:
        vec = litellm.embedding(model=embedding_model, input=["connectivity test"]).data[0]["embedding"]
        embedding_ok = True
        print(f"✅ {embedding_model} works (vector dimension {len(vec)})")
    except Exception as e:
        print(f"❌ {embedding_model} failed: {type(e).__name__}: {str(e)[:300]}")
        print("   Fix the key / OLLAMA_API_BASE in .env and re-run this cell, or set INGEST_EMBEDDINGS = False above to continue.")
USE_EMBEDDINGS = INGEST_EMBEDDINGS and embedding_ok
print("Will ingest with embeddings:", USE_EMBEDDINGS)

## 2 · Check the physical layer is there

The semantic layer *attaches to* the tables and columns notebook 01 created, so they must exist. We also take a **snapshot** of them now, to compare after the ingest.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(os.environ["NEO4J_URI"],
                              auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))

def q(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(cypher, **params).data()

before = q("""
MATCH (:Schema {name: $schema})-[:HAS_TABLE]->(t:Table)
OPTIONAL MATCH (t)-[:HAS_COLUMN]->(c:Column)
RETURN t.id AS id, t.name AS name, t.description AS description, count(c) AS columns
ORDER BY id
""", schema=SF_SCHEMA)

if not before:
    raise SystemExit(f"❌ No tables found for schema {SF_SCHEMA}. Run 01_load_metadata_into_graph.ipynb first, then come back.")

print(f"✅ Physical layer found: {len(before)} tables, {sum(t['columns'] for t in before)} columns")
for t in before:
    print(f"  {t['name']:<10}{t['columns']:>3} columns   {t['description']}")

def counts():
    # Node counts by label, relationship counts by type
    nodes = {r["label"]: r["n"] for r in q("MATCH (n) UNWIND labels(n) AS label RETURN label, count(*) AS n")}
    rels = {r["rel"]: r["n"] for r in q("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n")}
    return nodes, rels

nodes_before, rels_before = counts()
col_desc_before = {r["id"]: r["description"] for r in q("MATCH (c:Column) RETURN c.id AS id, c.description AS description")}
print("\nSemantic nodes already present:", {k: v for k, v in nodes_before.items() if k in ("OsiSemanticModel", "Metric", "Join", "BusinessTerm")} or "none")

## 3 · The semantic layer file

`datasets/tpch_semantic_model.yaml` is a business-facing description of the same eight tables, written by hand in the [Open Semantic Interchange](https://github.com/open-semantic-interchange/OSI) (OSI) format. Snowflake itself does not need to know about it: neocarta reads the file and writes it into the graph. It contains:

| OSI element | What it says | Becomes in the graph |
|---|---|---|
| `datasets` | Each table, its business description, primary key and synonyms | the **existing** `Table` nodes (extra label `OsiTable`) |
| `fields` | Each column with its meaning, synonyms and time-dimension flag | the **existing** `Column` nodes (extra label `OsiColumn`) |
| `relationships` | The foreign keys TPC-H does not declare | `Join` nodes and `REFERENCES` edges between columns |
| `metrics` | Business measures such as *net revenue*, each with one SQL formula | `Metric` + `Expression` nodes |
| `ai_context.synonyms` | Words the business uses: "purchases" for orders | `BusinessTerm` nodes, `TAGGED_WITH` the table, column or metric |

The key is each dataset's `source`: a 3-part name `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS`. neocarta turns it into the same node id notebook 01 used (`snowflake_sample_data.tpch_sf1.orders`), so the semantic layer lands on the physical nodes.

In [ ]:
spec = yaml.safe_load(SPEC.read_text())
model = spec["semantic_model"][0]
print(f"Model '{model['name']}' (OSI {spec['version']})")
print(f"  {len(model['datasets'])} datasets, {sum(len(d['fields']) for d in model['datasets'])} fields, "
      f"{len(model['relationships'])} relationships, {len(model['metrics'])} metrics\n")

orders = next(d for d in model["datasets"] if d["name"] == "ORDERS")
print("One dataset, trimmed:")
print(yaml.safe_dump({k: orders[k] for k in ("name", "source", "primary_key", "description", "ai_context")}, sort_keys=False, width=100))
print("One metric:")
print(yaml.safe_dump([m for m in model["metrics"] if m["name"] == "net_revenue"], sort_keys=False, width=100))

## 4 · Dry run, then ingest ✍️

The dry run prints the plan and the settings neocarta resolved, and touches nothing. We use the same `neocarta` CLI as in a terminal; the notebook passes the `.env` values through the environment.

In [ ]:
NEOCARTA = str(Path(sys.executable).parent / "neocarta")
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
SPINNER = re.compile(r"^[⠀-⣿]\s")   # braille spinner frames

def run_neocarta(*args, show=True):
    # Run the neocarta CLI with the loaded .env values; return (rc, cleaned output)
    env = {**os.environ, "NO_COLOR": "1", "TERM": "dumb"}
    proc = subprocess.run([NEOCARTA, *args], env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = [ANSI.sub("", l).rstrip() for l in proc.stdout.splitlines()]
    lines = [l for l in lines if l.strip() and not SPINNER.match(l)]
    text = "\n".join(lines)
    if show:
        print(text)
    return proc.returncode, text

rc, _ = run_neocarta("osi", "ingest", "--spec-source", str(SPEC), "--dry-run")
print("\n✅ Dry run OK" if rc == 0 else f"\n❌ Dry run failed (exit {rc}): fix the message above, then re-run")

In [ ]:
args = ["osi", "ingest", "--spec-source", str(SPEC)]
if USE_EMBEDDINGS:
    args += ["--embeddings"]
rc, out = run_neocarta(*args)
print("\n✅ Ingest succeeded" if rc == 0 else f"\n❌ Ingest failed (exit {rc}): see the message above")
assert rc == 0

## 5 · Did it map onto the physical layer?

If the semantic layer had created its own tables, we would now see 16 `Table` nodes. Instead the same 8 should carry an extra `OsiTable` label, and their names and descriptions from notebook 01 should be untouched.

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append(ok)
    print(("✅" if ok else "❌"), name, ("— " + detail) if detail else "")

nodes_after, rels_after = counts()
n_before = len(before)

check("No duplicate tables", nodes_after.get("Table", 0) == nodes_before.get("Table", 0), f"{nodes_after.get('Table')} Table nodes, same as before")
check("No duplicate columns", nodes_after.get("Column", 0) == nodes_before.get("Column", 0), f"{nodes_after.get('Column')} Column nodes, same as before")
check("Same Database / Schema", all(nodes_after.get(l) == nodes_before.get(l) for l in ("Database", "Schema")))

lab = q("MATCH (t:Table) RETURN count(t) AS tables, count(CASE WHEN t:OsiTable THEN 1 END) AS osi_tables")[0]
check("Existing tables carry the OsiTable label", lab["tables"] == lab["osi_tables"] > 0, f"{lab['osi_tables']} of {lab['tables']}")
lab = q("MATCH (c:Column) RETURN count(c) AS cols, count(CASE WHEN c:OsiColumn THEN 1 END) AS osi_cols")[0]
check("Existing columns carry the OsiColumn label", lab["cols"] == lab["osi_cols"] > 0, f"{lab['osi_cols']} of {lab['cols']}")

after = {t["id"]: t for t in q("MATCH (:Schema {name: $schema})-[:HAS_TABLE]->(t:Table) RETURN t.id AS id, t.name AS name, t.description AS description", schema=SF_SCHEMA)}
check("Table names and descriptions preserved", all(after[t["id"]]["name"] == t["name"] and after[t["id"]]["description"] == t["description"] for t in before))
col_desc_after = {r["id"]: r["description"] for r in q("MATCH (c:Column) RETURN c.id AS id, c.description AS description")}
check("Column descriptions preserved", col_desc_after == col_desc_before)

print("\nWhat the semantic layer added:")
for label in ("OsiSemanticModel", "Metric", "Join", "Expression", "BusinessTerm", "OsiAiContext"):
    print(f"  {label:<18}{nodes_before.get(label, 0):>4} → {nodes_after.get(label, 0):>4}")
for rel in ("REFERENCES", "TAGGED_WITH", "USED_IN_JOIN", "HAS_METRIC"):
    print(f"  {rel:<18}{rels_before.get(rel, 0):>4} → {rels_after.get(rel, 0):>4}")
check("Semantic nodes were added", nodes_after.get("Metric", 0) > 0 and nodes_after.get("Join", 0) > 0 and rels_after.get("REFERENCES", 0) > 0)

print("\n🎉 Semantic layer is attached to the physical layer." if all(checks) else "\n⚠️  Some checks failed: scroll up, fix, and re-run section 4.")

> **Why are the descriptions unchanged?** The OSI loader only sets a property when it *creates* a node. Nodes from notebook 01 already exist, so their `name` and `description` stay as Snowflake reported them, and the business meaning arrives as **new nodes connected to them** (below). On an empty graph the loader would also write the YAML descriptions onto the tables and columns.

## 6 · What the semantic layer adds

### Join paths (the foreign keys Snowflake does not declare)

Notebook 01 found no `REFERENCES` edges because TPC-H declares no keys. The `relationships` in the YAML supply them. Each row is a column-to-column edge between two physical tables:

In [ ]:
import pandas as pd

pd.DataFrame(q("""
MATCH (a:Table)-[:HAS_COLUMN]->(ca:Column)-[:REFERENCES]->(cb:Column)<-[:HAS_COLUMN]-(b:Table)
RETURN a.name AS from_table, ca.name AS foreign_key, b.name AS to_table, cb.name AS references_column
ORDER BY from_table, foreign_key
"""))

Because the edges are in the graph, a join route is a path query. How does an order line reach a region?

> `LINEITEM → PARTSUPP` appears twice because its composite foreign key `(L_PARTKEY, L_SUPPKEY)` is stored as one edge per column pair.

In [ ]:
route = q("""
MATCH p = shortestPath((a:Table {name: 'LINEITEM'})-[:HAS_COLUMN|REFERENCES*..20]-(b:Table {name: 'REGION'}))
RETURN [c IN nodes(p) WHERE c:Column | [(t:Table)-[:HAS_COLUMN]->(c) | t.name][0] + '.' + c.name] AS hops
""")
print("LINEITEM → REGION, column by column:")
print("  " + "\n  → ".join(route[0]["hops"]))

### Business-term synonyms

Every `synonyms` list became `BusinessTerm` nodes tagged onto the tables, columns and metrics they describe. Someone asking about *"purchases"* or *"revenue"* uses words that appear nowhere in Snowflake's column names:

In [ ]:
TERM = "revenue"
pd.DataFrame(q("""
MATCH (bt:BusinessTerm)<-[:TAGGED_WITH]-(x)
WHERE toLower(bt.name) IN [toLower($term), 'purchases', 'discount rate', 'market segment']
RETURN bt.name AS business_term, labels(x)[0] AS tagged_node_type, x.name AS tagged_node
ORDER BY business_term, tagged_node
""", term=TERM))

### Metrics: one agreed formula per business measure

A metric carries its definition once, in the graph, instead of every analyst (or LLM) re-deriving "revenue" differently. The SQL is kept as an `Expression` node:

In [ ]:
pd.set_option("display.max_colwidth", 110)
pd.DataFrame(q("""
MATCH (m:Metric)-[:HAS_EXPRESSION]->(e:Expression)
RETURN m.name AS metric, m.description AS definition, e.expression AS sql
ORDER BY metric
"""))

## 7 · The agent's view

The MCP server (and the CLI mirror we use here) returns a *table context* for a question: tables, their columns, example values and relationships. The tool below looks tables up by business-glossary term, which only works because of the `BusinessTerm` nodes we just loaded. Note what each column carries: **`references`** (join targets) and **`aspects`** (the synonyms), both of which come from the semantic layer.

In [ ]:
QUESTION = "purchases"
cmd = ["tool", "get-context-by-table-business-term-hybrid-search", "--text-content", QUESTION, "--max-tables", "3", "--json"]
rc, out = run_neocarta(*cmd, show=False)
if rc != 0:
    print("❌ Search failed:\n", out[-800:])
else:
    # The CLI may print driver notices before the JSON document: parse from the first line that starts it
    start = re.search(r"^\{", out, re.M).start()
    results = next(iter(json.loads(out[start:]).values()))["results"]
    rows = []
    for t in results:
        cols = t["columns"]
        rows.append({
            "table": t["table_name"],
            "columns": len(cols),
            "with join references": sum(1 for c in cols if c.get("references")),
            "with business-term aspects": sum(1 for c in cols if c.get("aspects")),
        })
    print(f"Tables returned for '{QUESTION}' (the business term 'purchases' is tagged on ORDERS; the other hits are vector neighbours):")
    display(pd.DataFrame(rows))
    check("ORDERS found through the business term", any(r["table"] == "ORDERS" for r in rows))
    sample = next((c for t in results if t["table_name"] == "ORDERS" for c in t["columns"] if c["column_name"] == "O_CUSTKEY"), None)
    if sample:
        print("Example column the agent receives (ORDERS.O_CUSTKEY):")
        print(json.dumps({k: sample[k] for k in ("column_name", "data_type", "references", "expressions")}, indent=2))

## 8 · Re-run safely

Ingestion is idempotent: running the same ingest again must not add anything.

In [ ]:
nodes_1, rels_1 = counts()
rc, _ = run_neocarta("osi", "ingest", "--spec-source", str(SPEC), *(["--embeddings"] if USE_EMBEDDINGS else []), show=False)
nodes_2, rels_2 = counts()
check("Second ingest succeeded", rc == 0)
check("No nodes or relationships duplicated", (nodes_1, rels_1) == (nodes_2, rels_2))

## Summary and next steps

The graph now holds two layers joined on the same nodes:

| Layer | Source | Answers |
|---|---|---|
| Physical (notebook 01) | Snowflake `INFORMATION_SCHEMA` | *Where* is the data, what types, what values? |
| Semantic (this notebook) | Business model YAML (OSI) | *What does it mean*, how do tables join, how is revenue defined? |

Next:

- **Chat with it:** `uv run workshop-agent` (see `docs/attendee_guide.md` section 7.5): ask for *"net revenue by region"* and watch it use the join path and the metric definition.
- **Make it yours:** copy `datasets/tpch_semantic_model.yaml`, describe your own tables, and ingest it with `neocarta osi ingest --spec-source <file>`.
- **Round trip:** `neocarta osi export` writes the semantic model back out of the graph as OSI YAML.

In [ ]:
driver.close()